# Design fixation among language model agents: analysis notebook
# E.Brubaker, P.Saha

Generates figures from the consolidated CSV files in `data/`.

| Figure | Data file(s) |
|---|---|
| Fig. 2 Study 1, multi-example sessions | `study1_multi_example_runs.csv` |
| Fig. 3 Study 1, single-example test | `study1_single_example_concepts.csv` |
| Fig. 4 Study 2, teams | `study2_team_concepts.csv`, `study2_dose_response.csv` |
| Fig. 5 Study 3, heatmaps | `study3_model_profiles.csv` |
| Fig. 6 Study 3, own default and ultrasonic shown | `study3_exposure_scan.csv` |

Outputs go to `figures/` as PDF (vector) and PNG (preview).

In [ ]:
import json, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

DATA = "data"; OUT = "figures"
PALETTE = "tol_muted"   # "tol_muted" or "okabe_ito"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5,
    "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "savefig.bbox": "tight", "savefig.dpi": 300, "pdf.fonttype": 42,
})


PALETTES = {
    "okabe_ito": dict(primary="#0072B2", secondary="#D55E00", tertiary="#E69F00", light="#56B4E9",
                      green="#009E73", accent="#CC79A7", grey="#8C8C8C"),
    "tol_muted": dict(primary="#332288", secondary="#CC6677", tertiary="#999933", light="#88CCEE",
                      green="#117733", accent="#AA4499", grey="#8C8C8C"),
}
PAL = PALETTES[PALETTE]

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0, c-h), min(1, c+h)

def boot_ci(x, B=3000, seed=0):
    x = np.asarray(x, float); rng = np.random.default_rng(seed)
    m = rng.choice(x, (B, len(x))).mean(1)
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

def boot_diff(x, y, B=3000, seed=0):
    x = np.asarray(x, float); y = np.asarray(y, float); rng = np.random.default_rng(seed)
    d = rng.choice(x, (B, len(x))).mean(1) - rng.choice(y, (B, len(y))).mean(1)
    return x.mean()-y.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def panel_label(ax, s):
    ax.text(-0.02, 1.06, s, transform=ax.transAxes, ha="right", fontweight="bold", fontsize=9, va="bottom")

def save(fig, name):
    import os; os.makedirs(OUT, exist_ok=True)
    fig.savefig(f"{OUT}/{name}.pdf"); fig.savefig(f"{OUT}/{name}.png"); print("saved", name)


## Figure 2. Study 1, multi-example sessions

In [2]:
u = pd.read_csv(f"{DATA}/study1_multi_example_runs.csv")
fig, axs = plt.subplots(1, 2, figsize=(5.6, 2.3), gridspec_kw=dict(width_ratios=[1.15, 1], wspace=0.5))
ex = ["none", "one_example", "three_examples", "own_history"]; exlab = ["none", "1 ex.", "3 ex.", "own\nmemory"]
col = {"neutral": PAL["primary"], "verbatim": PAL["secondary"]}
ax = axs[0]
for i, b in enumerate(["neutral", "verbatim"]):
    base = u[(u.brief == b) & (u.exposure == "none")].example_features_per_concept
    for j, e in enumerate(ex):
        x = u[(u.brief == b) & (u.exposure == e)].example_features_per_concept
        m, lo, hi = boot_diff(x, base, seed=j)
        ax.bar(j + (i-0.5)*0.36, m, 0.36, color=col[b], label=b if j == 0 else None)
        ax.errorbar(j + (i-0.5)*0.36, m, [[m-lo], [hi-m]], color="k", lw=0.7, capsize=2)
ax.axhline(0, color="k", lw=0.5); ax.set_xticks(range(4)); ax.set_xticklabels(exlab)
ax.set_ylabel("excess exemplar features\nper concept (vs. unexposed)"); ax.legend(frameon=False, title="brief", title_fontsize=7)
ax.set_title("Excess borrowing", loc="left", x=0.02); panel_label(ax, "(a)")
feats = [("heater_share", "integrated heater", PAL["tertiary"]),
         ("magnetic_coupling_share", "magnetic coupling", PAL["green"]),
         ("venturi_share", "venturi nozzle", PAL["light"]),
         ("flaw_share", "app-only flaw", PAL["accent"])]
ax = axs[1]; b = "neutral"
for f, lab, c in feats:
    ms = [u[(u.brief == b) & (u.exposure == e)][f].mean() for e in ex[:3]]
    ax.plot(range(3), ms, marker="o", color=c, ms=4, ls="none", label=lab)
    mm = u[(u.brief == b) & (u.exposure == "own_history")][f].mean()
    ax.plot([3.6], [mm], marker="D", color=c, ms=3.8, ls="none")
ax.axvline(3.3, color="0.6", lw=0.6, ls=":")
ax.set_xticks([0, 1, 2, 3.6]); ax.set_xticklabels(exlab); ax.set_ylim(-0.005, 0.16); ax.set_xlim(-0.3, 4.0)
ax.set_ylabel("share of concepts with feature"); ax.set_title("By feature (neutral brief)", loc="left", x=0.02)
panel_label(ax, "(b)"); ax.legend(frameon=False, loc="upper left", fontsize=6.5, handletextpad=0.2)
save(fig, "fig2_study1")


saved fig2_study1


## Figure 3. Study 1, single-example test

In [13]:
d = pd.read_csv(f"{DATA}/study1_single_example_concepts.csv")
P = {m: d[d.model == m] for m in ["Llama-3.1-8B", "Llama-3.3-70B"]}
mk = {"Llama-3.1-8B": ("o", PAL["primary"]), "Llama-3.3-70B": ("s", PAL["secondary"])}
xs = ["baseline", "neutral", "permissive", "directive"]
rc = {"font.size": 9.5, "axes.titlesize": 10, "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5}
with plt.rc_context(rc):
    fig = plt.figure(figsize=(8.0, 7.0))
    gs = GridSpec(2, 3, figure=fig, hspace=0.78, wspace=0.42, width_ratios=[1, 1, 0.62])
    axs = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :2])]
    def share(dd, shown, instr, col):
        s = dd[dd.example_shown == "none"] if instr == "baseline" else dd[(dd.example_shown == shown) & (dd.instruction == instr)]
        return wilson(int(s[col].sum()), len(s))
    for k, (shown, col, title) in enumerate([("whisk", "has_whisk", "Fixation on whisk when\nprovided a whisk example"),
                                             ("venturi", "has_venturi", "Fixation on Venturi when\nprovided a Venturi example")]):
        ax = axs[k]
        for j, (m, dd) in enumerate(P.items()):
            vals = [share(dd, shown, x, col) for x in xs]
            xx = np.arange(4) + (j-0.5)*0.16
            ax.errorbar(xx, [v[0] for v in vals], [[v[0]-v[1] for v in vals], [v[2]-v[0] for v in vals]],
                        marker=mk[m][0], color=mk[m][1], ms=6, lw=1.3, ls="none", capsize=2.5, label=m)
        ax.set_xticks(range(4)); ax.set_xticklabels(["Unexposed", "Neutral prompt", "Permissive prompt", "Directive prompt"], rotation=30, ha="right", rotation_mode="anchor")
        ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.03, 1.12); ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.set_ylabel("Share of concepts"); ax.set_title(title, loc="left", x=0.0, pad=12)
        ax.axvline(0.5, color="0.6", lw=0.6, ls=":")
        ax.text(-0.02, 1.26, "(a)" if k == 0 else "(b)", transform=ax.transAxes, ha="right", fontweight="bold", fontsize=10, va="bottom")
    axs[1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.06, 1.0), ncol=1, handletextpad=0.3, title="Key", title_fontsize=8.5, alignment="left")
    ax = axs[2]
    for j, (m, dd) in enumerate(P.items()):
        enacted = []; stated = []
        for ins in ["neutral", "permissive", "directive"]:
            a = dd[(dd.example_shown == "whisk") & (dd.instruction == ins)].has_whisk
            b = dd[(dd.example_shown == "venturi") & (dd.instruction == ins)].has_venturi
            enacted.append(wilson(int(a.sum() + b.sum()), len(a) + len(b)))
            r = dd[(dd.example_shown != "none") & (dd.instruction == ins)].copy_language
            stated.append(wilson(int(r.sum()), len(r)))
        for vals, mfc, lab, dx in [(enacted, mk[m][1], f"{m}:\nEnacted - adopted feature\nfrom provided example", -0.09),
                                   (stated, "white", f"{m}:\nStated - agent's rationale\ndescribed adopting feature\nfrom provided example", 0.09)]:
            xx = np.arange(3) + dx + (j-0.5)*0.30
            ax.errorbar(xx, [v[0] for v in vals], [[v[0]-v[1] for v in vals], [v[2]-v[0] for v in vals]],
                        marker=mk[m][0], color=mk[m][1], ms=6, lw=1.2, ls="none", mfc=mfc, capsize=2.5, label=lab)
    ax.set_xticks(range(3)); ax.set_xticklabels(["Neutral prompt", "Permissive prompt", "Directive prompt"])
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.03, 1.03); ax.set_ylabel("Share of rationales or concepts")
    ax.set_title("Stated versus enacted feature adoption\n(for combined whisk and Venturi examples)", loc="left", x=0.0, pad=12)
    ax.text(-0.02, 1.26, "(c)", transform=ax.transAxes, ha="right", fontweight="bold", fontsize=10, va="bottom")
    ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.03, 1.30), labelspacing=1.0, handletextpad=0.4, title="Key", title_fontsize=8.5, alignment="left")
    save(fig, "fig3_study2")


saved fig3_study2


## Combined figure: Studies 1 and 3, single-agent response to examples 

In [ ]:

d1 = pd.read_csv(f"{DATA}/study1_single_example_concepts.csv")
S3 = pd.read_csv(f"{DATA}/study3_exposure_scan.csv")
Pm3 = pd.read_csv(f"{DATA}/study3_model_profiles.csv")
labs3 = list(Pm3.sort_values("model_order").model.unique())
own3 = S3[S3.is_own_default_mechanism == 1].groupby("model").example_feature.first().to_dict()
P1 = {m: d1[d1.model == m] for m in ["Llama-3.1-8B", "Llama-3.3-70B"]}
mk = {"Llama-3.1-8B": ("o", PAL["primary"]), "Llama-3.3-70B": ("s", PAL["secondary"])}
xs = ["baseline", "neutral", "permissive", "directive"]
FS = dict(panel=9.5, title=8.5, label=8, tick=7.5, cat=7, legend=7, key_title=8)
rc = {"font.size": FS["label"], "axes.labelsize": FS["label"], "xtick.labelsize": FS["tick"],
      "ytick.labelsize": FS["tick"], "legend.fontsize": FS["legend"], "savefig.bbox": "standard"}
W, H = 5.15, 7.0
KEYBOX = dict(frameon=True, fancybox=False, edgecolor="0.8", facecolor="white", framealpha=1.0)

with plt.rc_context(rc):
    fig = plt.figure(figsize=(W, H))
    def box(l, b, w, h):                       # position in inches from the canvas's bottom-left
        return fig.add_axes([l / W, b / H, w / W, h / H])
    def head(ax, letter, title, gap=0.06, lx=None):
        """Panel letter and title above an axis; the title's first line is level with the letter."""
        pos = ax.get_position(); top = pos.y1 * H + gap
        left = pos.x0 * W if lx is None else lx
        fig.text((left + 0.27) / W, top / H, title, ha="left", va="bottom", fontsize=FS["title"], linespacing=1.15)
        nlines = title.count("\n")
        fig.text(left / W, (top + nlines * FS["title"] * 1.15 / 72) / H, letter, ha="left", va="bottom",
                 fontsize=FS["panel"], fontweight="bold")

    axa = box(0.55, 5.55, 2.05, 0.90)
    axb = box(2.97, 5.55, 2.05, 0.90)
    axc = box(1.72, 2.62, 1.55, 1.92)
    axd = box(3.47, 2.62, 1.55, 1.92)
    axe = box(0.62, 0.40, 2.30, 0.88)

    # (a, b) Study 1 single-example test
    def share(dd, shown, instr, col):
        s = dd[dd.example_shown == "none"] if instr == "baseline" else dd[(dd.example_shown == shown) & (dd.instruction == instr)]
        return wilson(int(s[col].sum()), len(s))
    for ax, (shown, col) in [(axa, ("whisk", "has_whisk")), (axb, ("venturi", "has_venturi"))]:
        for j, (m, dd) in enumerate(P1.items()):
            vals = [share(dd, shown, x, col) for x in xs]; xx = np.arange(4) + (j-0.5)*0.2
            ax.errorbar(xx, [v[0] for v in vals], [[v[0]-v[1] for v in vals], [v[2]-v[0] for v in vals]],
                        marker=mk[m][0], color=mk[m][1], ms=4.5, lw=1.0, ls="none", capsize=1.8, label=m)
        ax.set_xticks(range(4))
        ax.set_xticklabels(["Unexposed", "Neutral\nprompt", "Permissive\nprompt", "Directive\nprompt"], fontsize=FS["cat"])
        ax.set_xlim(-0.55, 3.55); ax.set_ylim(-0.04, 1.06); ax.set_yticks([0, 0.5, 1]); ax.set_yticklabels(["0", "0.5", "1"])
    axa.set_ylabel("Share of concepts")
    head(axa, "(a)", "Fixation on whisk when\nprovided a whisk example", lx=0.05)
    head(axb, "(b)", "Fixation on Venturi when\nprovided a Venturi example", lx=2.67)
    lb = axb.legend(**KEYBOX, loc="upper left", bbox_to_anchor=(0.04, 0.99), handletextpad=0.2, borderaxespad=0, borderpad=0.4,
               labelspacing=0.3, title="Key", title_fontsize=FS["key_title"], alignment="left")

    # (c, d) Study 3 exposure scan, twelve models
    y = np.arange(len(labs3))[::-1]
    def g_feat(l, f):
        s = S3[(S3.model == l) & (S3.example_feature == f)]
        return s.unexposed_share.iloc[0], s[s.instruction == "neutral"].shown_share.iloc[0], s[s.instruction == "directive"].shown_share.iloc[0]
    for ax, getter in [(axc, lambda l: g_feat(l, own3[l])), (axd, lambda l: g_feat(l, "ultrasonic"))]:
        for yi, l in zip(y, labs3):
            b, n, dd = getter(l)
            ax.plot([b, n], [yi, yi], color="0.75", lw=0.8, zorder=1)
            ax.plot([b, dd], [yi, yi], color="0.75", lw=0.8, ls=":", zorder=1)
            ax.scatter(n, yi, s=18, color=PAL["secondary"], marker="v", zorder=3, lw=0,
                       label="Example provided,\nNeutral prompt" if yi == y[0] else None)
            ax.scatter(b, yi, s=16, color=PAL["grey"], zorder=3, lw=0, label="Unexposed (no\nexample provided)" if yi == y[0] else None)
            ax.scatter(dd, yi, s=18, color=PAL["primary"], marker="^", zorder=3, lw=0,
                       label="Example provided,\nDirective prompt" if yi == y[0] else None)
        ax.set_xlim(-0.06, 1.06); ax.set_xticks([0, 0.5, 1]); ax.set_xticklabels(["0", "0.5", "1"])
        ax.set_ylim(-0.6, len(labs3) - 0.4); ax.set_xlabel("Share of concepts", labelpad=2)
        ax.grid(axis="x", color="0.92", lw=0.5); ax.set_axisbelow(True)
    axc.set_yticks(y); axc.set_yticklabels([f"{l} [{own3[l].split('_')[0]}]" for l in labs3], fontsize=FS["cat"])
    axd.set_yticks(y); axd.set_yticklabels([])
    head(axc, "(c)", "Fixation when provided model’s\n“default” frothing mechanism", lx=0.05)
    head(axd, "(d)", "Fixation on ultrasonic when\nprovided an ultrasonic example", lx=2.67)
    h, l_ = axd.get_legend_handles_labels()
    leg = fig.legend(h, l_, frameon=False, loc="center right", bbox_to_anchor=(4.98 / W, 2.03 / H), ncol=3,
                     columnspacing=1.6, handletextpad=0.15, borderaxespad=0, borderpad=0)
    fig.canvas.draw()
    ext = leg.get_window_extent(); lx0 = ext.x0 / fig.dpi   # left edge of the key, inches
    kt = fig.text((lx0 - 0.08) / W, 2.03 / H, "Key", fontsize=FS["key_title"], va="center", ha="right")
    fig.canvas.draw()
    from matplotlib.patches import Rectangle
    x0 = kt.get_window_extent().x0 / fig.dpi - 0.08; x1 = ext.x1 / fig.dpi + 0.08
    y0 = ext.y0 / fig.dpi - 0.06; y1 = ext.y1 / fig.dpi + 0.06
    fig.patches.append(Rectangle((x0 / W, y0 / H), (x1 - x0) / W, (y1 - y0) / H, transform=fig.transFigure,
                                 fill=False, edgecolor="0.8", lw=0.8, zorder=0))

    # (e) stated vs enacted, Study 1
    ax = axe
    for j, (m, dd) in enumerate(P1.items()):
        enacted = []; stated = []
        for ins in ["neutral", "permissive", "directive"]:
            a = dd[(dd.example_shown == "whisk") & (dd.instruction == ins)].has_whisk
            b = dd[(dd.example_shown == "venturi") & (dd.instruction == ins)].has_venturi
            enacted.append(wilson(int(a.sum() + b.sum()), len(a) + len(b)))
            r = dd[(dd.example_shown != "none") & (dd.instruction == ins)].copy_language
            stated.append(wilson(int(r.sum()), len(r)))
        for vals, mfc, lab, dx in [(enacted, mk[m][1], f"{m}: Enacted - adopted\nfeature from provided example", -0.09),
                                   (stated, "white", f"{m}: Stated - agent's\nrationale described adopting\nfeature from provided example", 0.09)]:
            xx = np.arange(3) + dx + (j-0.5)*0.30
            ax.errorbar(xx, [v[0] for v in vals], [[v[0]-v[1] for v in vals], [v[2]-v[0] for v in vals]],
                        marker=mk[m][0], color=mk[m][1], ms=4.5, lw=1.0, ls="none", mfc=mfc, capsize=1.8, label=lab)
    ax.set_xticks(range(3)); ax.set_xticklabels(["Neutral\nprompt", "Permissive\nprompt", "Directive\nprompt"], fontsize=FS["cat"])
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.04, 1.06); ax.set_yticks([0, 0.5, 1]); ax.set_yticklabels(["0", "0.5", "1"])
    ax.set_ylabel("Share of rationales\nor concepts", linespacing=1.0, fontsize=FS["cat"] + 0.5)
    head(ax, "(e)", "Stated versus enacted feature adoption\n(for whisk and Venturi examples)", lx=0.05)
    le = ax.legend(**KEYBOX, loc="upper left", bbox_to_anchor=(1.05, 1.47), labelspacing=0.45, handletextpad=0.25, borderpad=0.5,
              title="Key", title_fontsize=FS["key_title"], alignment="left")
    save(fig, "fig1_combined")


saved fig1_combined


## Figure 4. Study 2, eight-agent teams

In [5]:
T0 = pd.read_csv(f"{DATA}/study2_team_concepts.csv")
T0["F"] = T0.features.fillna("").apply(lambda s: set(s.split(";")) - {""})
def team_stats(g):
    ab = risk = 0; jac = []
    for a, ga in g.groupby("agent"):
        ga = ga.sort_values("round"); F = list(ga.F)
        for t in range(len(F)-1):
            risk += len(F[t]); ab += len(F[t] - F[t+1])
            u_ = F[t] | F[t+1]; jac.append(len(F[t] & F[t+1]) / len(u_) if u_ else np.nan)
    return ab / risk, np.nanmean(jac)
rows = []
D = T0[T0.source_job == "D"]; D2 = T0[T0.source_job == "D2"]
for (ins, unit), g in D.groupby(["instruction", "team_id"]):
    gm, j = team_stats(g); rows.append(dict(cond=ins, gamma=gm, jac=j))
for (vis, unit), g in D2.groupby(["visible_teammates", "team_id"]):
    gm, j = team_stats(g); rows.append(dict(cond=f"D2_see{vis}", gamma=gm, jac=j))
Tst = pd.DataFrame(rows)
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.35), gridspec_kw=dict(width_ratios=[0.8, 1.2, 1.1], wspace=0.45))
ax = axs[0]
for i, (c, col) in enumerate([("propose_next", PAL["grey"]), ("revise", PAL["primary"])]):
    m, lo, hi = boot_ci(Tst[Tst.cond == c].gamma)
    ax.bar(i, m, 0.6, color=col); ax.errorbar(i, m, [[m-lo], [hi-m]], color="k", lw=0.7, capsize=2)
ax.set_xticks([0, 1]); ax.set_xticklabels(["'propose your\nnext concept'", "'revise or\nreplace'"]); ax.set_ylim(0, 0.6)
ax.set_ylabel("abandonment rate $\\gamma$\n(all features, per agent-round)"); ax.set_title("Abandonment", loc="left", x=0.02); panel_label(ax, "(a)")
ax = axs[1]
def prev(dd, feat="temperature_control"):
    dd = dd.assign(h=dd.F.apply(lambda s: feat in s))
    team = dd.groupby(["team_id", "round"]).h.mean().reset_index()
    return team.groupby("round").h.apply(lambda x: boot_ci(x)).apply(pd.Series)
curves = [(D2[D2.visible_teammates == 0], "revise, work alone", PAL["grey"], "-"),
          (D2[D2.visible_teammates == 3], "revise, see 3 teammates", PAL["light"], "-"),
          (D2[D2.visible_teammates == 7], "revise, see all 7", PAL["primary"], "-"),
          (D[(D.instruction == "propose_next") & (D.planted == 0)], "propose next, see all 7", PAL["tertiary"], "--")]
for dd, lab, col, ls in curves:
    p = prev(dd); ax.plot(p.index, p[0], color=col, ls=ls, lw=1.3, marker="o", ms=2.5, label=lab)
    ax.fill_between(p.index, p[1], p[2], color=col, alpha=0.12, lw=0)
ax.set_xlabel("round"); ax.set_ylabel("share of concepts with\ntemperature control"); ax.set_ylim(0, 1)
ax.set_title("Spread of an unseeded feature", loc="left", x=0.02); ax.legend(frameon=False, loc="upper left", fontsize=6.3); panel_label(ax, "(b)")
ax = axs[2]
dr_tab = pd.read_csv(f"{DATA}/study2_dose_response.csv")
bins = ["0", "1", "2", "3-4", "5+"]
def dr(job, conds):
    # NOTE: the manuscript figure pools the planted and unplanted 'propose next' teams for this panel,
    # exactly as the original script did (its cell filter matched both). Pass a list of conditions to pool.
    s = dr_tab[(dr_tab.feature == "temperature_control") & (dr_tab.source_job == job) & (dr_tab.condition.isin(conds))].copy(); s["k"] = s.adoption_prob * s.n_agent_rounds
    g = s.groupby("visible_carriers_bin")[["k", "n_agent_rounds"]].sum()
    return [wilson(round(g.loc[b, "k"]), int(g.loc[b, "n_agent_rounds"])) if b in g.index and g.loc[b, "n_agent_rounds"] >= 20 else (np.nan,)*3 for b in bins]
for vals, lab, col, ls, dx in [(dr("D2", ["revise_see7"]), "revise, see all 7", PAL["primary"], "-", -0.08),
                                (dr("D2", ["revise_see3"]), "revise, see 3", PAL["light"], "-", 0.0),
                                (dr("D", ["propose_next_see7", "propose_next_see7_planted"]), "propose next, see all 7", PAL["tertiary"], "--", 0.08)]:
    x = np.arange(5) + dx; m = np.array([v[0] for v in vals])
    ax.errorbar(x, m, [m - np.array([v[1] for v in vals]), np.array([v[2] for v in vals]) - m], color=col, ls=ls, marker="o", ms=3, lw=1.1, capsize=1.5, label=lab)
ax.set_xticks(range(5)); ax.set_xticklabels(bins); ax.set_xlabel("visible teammates expressing it")
ax.set_ylabel("P(adopt next round | absent)"); ax.set_ylim(0, 0.8)
ax.set_title("Dose-response", loc="left", x=0.02); ax.legend(frameon=False, fontsize=6.3, loc="upper left"); panel_label(ax, "(c)")
save(fig, "fig4_teams")


saved fig4_teams


## Figure 5. Study 3, defaults and social index across twelve models

In [23]:
Pm = pd.read_csv(f"{DATA}/study3_model_profiles.csv")
labs = list(Pm.sort_values("model_order").model.unique())
feat_order = list(Pm[Pm.model == labs[0]].feature)   # schema order
Dm = Pm.pivot(index="feature", columns="model", values="default").loc[feat_order, labs]
SI = Pm.pivot(index="feature", columns="model", values="social_index").loc[feat_order, labs]
keep = [f for f in Dm.index if Dm.loc[f].max() >= 0.3 or SI.loc[f].abs().max() >= 0.25]
order = [f for f in Dm.index if f in keep]
nice = {"whisk_coil": "whisk", "impeller_turbine": "impeller", "aerator_nozzle": "venturi nozzle", "ultrasonic": "ultrasonic",
        "centrifugal_disc": "spinning disc", "battery_motor": "battery motor", "usb_rechargeable": "USB battery",
        "magnetic_drive": "magnetic coupling", "silicone_body": "silicone body", "stainless_steel": "stainless steel",
        "glass_vessel": "glass vessel", "recycled_plastic": "recycled plastic", "modular_detachable": "modular parts",
        "double_wall_insulation": "double wall", "integrated_heating": "heater", "temperature_control": "temperature control",
        "one_touch_auto": "one-button", "auto_stop_sensing": "auto-stop", "froth_texture_setting": "froth-texture setting",
        "sight_window": "sight window", "handheld_wand": "handheld wand", "steam_injection": "steam"}
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6), gridspec_kw=dict(wspace=0.22))
for ax, Tm, cmap, vmin, vmax, title in [(axs[0], Dm, "Blues", 0, 1, "Default features: share of unexposed concepts\n"),
                                        (axs[1], SI, "RdBu_r", -0.6, 0.6, "Social attractors: difference in rise in adoption\n with teammates versus rise in adoption alone")]:
    im = ax.imshow(Tm.loc[order].values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=55, ha="right", fontsize=6.2)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([nice.get(f, f) for f in order] if ax is axs[0] else [], fontsize=6.5)
    ax.set_title(title, loc="left", fontsize=7.5)
    for i, f in enumerate(order):
        for j, l in enumerate(labs):
            v = Tm.loc[f, l]
            if abs(v) >= (0.5 if Tm is Dm else 0.3): ax.text(j, i, f"{v:.1f}".replace("0.", "."), ha="center", va="center", fontsize=4.8, color="white")
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02); cb.ax.tick_params(labelsize=6)
    for s in ax.spines.values(): s.set_visible(False)
panel_label(axs[0], "(a)"); panel_label(axs[1], "(b)")
save(fig, "fig5_models_heatmap")


/var/folders/26/z678m3713c51xq9bsb01qnyc0000gn/T/ipykernel_88632/2749621275.py:15: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6), gridspec_kw=dict(wspace=0.22))


saved fig5_models_heatmap


## Figure 6. Study 3, own default and ultrasonic

In [7]:
S = pd.read_csv(f"{DATA}/study3_exposure_scan.csv")
own = S[S.is_own_default_mechanism == 1].groupby("model").example_feature.first().to_dict()
fig, axs = plt.subplots(1, 2, figsize=(5.2, 3.7), sharey=True, gridspec_kw=dict(wspace=0.16))
y = np.arange(len(labs))[::-1]
def dumb(ax, getter, title):
    for yi, l in zip(y, labs):
        b, n, dd = getter(l)
        ax.plot([b, n], [yi, yi], color="0.75", lw=1, zorder=1)
        ax.plot([b, dd], [yi, yi], color="0.75", lw=1, ls=":", zorder=1)
        ax.scatter(b, yi, s=18, color=PAL["grey"], zorder=3, label="unexposed" if yi == y[0] else None)
        ax.scatter(n, yi, s=18, color=PAL["secondary"], marker="v", zorder=3, label="shown, neutral instruction" if yi == y[0] else None)
        ax.scatter(dd, yi, s=18, color=PAL["primary"], marker="^", zorder=3, label="shown, 'build directly on it'" if yi == y[0] else None)
    ax.set_xlim(-0.04, 1.04); ax.set_title(title, loc="left", x=0.0); ax.set_xlabel("share of concepts")
def g_feat(l, f):
    s = S[(S.model == l) & (S.example_feature == f)]
    return s.unexposed_share.iloc[0], s[s.instruction == "neutral"].shown_share.iloc[0], s[s.instruction == "directive"].shown_share.iloc[0]
dumb(axs[0], lambda l: g_feat(l, own[l]), "Own default shown"); panel_label(axs[0], "(a)")
dumb(axs[1], lambda l: g_feat(l, "ultrasonic"), "Ultrasonic shown"); panel_label(axs[1], "(b)")
axs[0].set_yticks(y); axs[0].set_yticklabels([f"{l} [{own[l].split('_')[0]}]" for l in labs])
h, l_ = axs[1].get_legend_handles_labels(); fig.legend(h, l_, frameon=False, fontsize=6.3, loc="lower center", bbox_to_anchor=(0.55, -0.04), ncol=3)
for a in axs: a.grid(axis="x", color="0.92", lw=0.5); a.set_axisbelow(True)
save(fig, "fig6_models_response")


saved fig6_models_response
